# Завдання 1

У цьому завданні потрібно завантажити датасет (2D датасет та MNIST), застосувати K-means для кластеризації та знайти оптимальну кількість кластерів за допомогою ліктевого методу.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

## Завантаження 2D датасету

In [ ]:
df_2d = pd.read_csv('data/data_2d.csv')
X_2d = df_2d.values

plt.figure(figsize=(6, 4))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c='steelblue', s=30)
plt.title('2D Dataset')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

print(f'Розмір 2D датасету: {X_2d.shape}')

## Ліктевий метод для 2D датасету

In [ ]:
inertia = []
K = range(1, 11)
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_2d)
    inertia.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(K, inertia, 'bo-')
plt.xlabel('Кількість кластерів')
plt.ylabel('Інерція')
plt.title('Ліктевий метод для 2D датасету')
plt.grid(True)
plt.show()

## Завантаження MNIST датасету

In [ ]:
df_mnist = pd.read_csv('data/mnist.csv')
X_mnist = df_mnist.drop(columns=['label']).values if 'label' in df_mnist.columns else df_mnist.values
y_mnist = df_mnist['label'].values if 'label' in df_mnist.columns else None

print(f'Розмір MNIST датасету: {X_mnist.shape}')
if y_mnist is not None:
    print(f'Класи: {np.unique(y_mnist)}')

## Ліктевий метод для MNIST датасету

In [ ]:
scaler = StandardScaler()
X_mnist_scaled = scaler.fit_transform(X_mnist)

inertia_mnist = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_mnist_scaled)
    inertia_mnist.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(range(1, 11), inertia_mnist, 'ro-')
plt.xlabel('Кількість кластерів')
plt.ylabel('Інерція')
plt.title('Ліктевий метод для MNIST')
plt.grid(True)
plt.show()

## Візуалізація результатів кластеризації

### 2D Dataset

In [ ]:
k_opt_2d = inertia.index(min(inertia[1:])) + 1 if len(inertia) > 1 else 4
kmeans_2d = KMeans(n_clusters=k_opt_2d, random_state=42, n_init=10)
y_pred_2d = kmeans_2d.fit_predict(X_2d)

plt.figure(figsize=(6, 4))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_pred_2d, cmap='tab10', s=30)
plt.scatter(kmeans_2d.cluster_centers_[:, 0], kmeans_2d.cluster_centers_[:, 1], 
            c='black', marker='x', s=200, linewidths=3, label='Центроїди')
plt.title(f'K-means кластеризація 2D датасету (k={k_opt_2d})')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.show()

### MNIST Dataset (з PCA)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_mnist_2d = pca.fit_transform(X_mnist_scaled)

print(f'Зменшена розмірність: {X_mnist_2d.shape}')
print(f'Пояснена дисперсія: {pca.explained_variance_ratio_.sum():.2%}')

In [ ]:
k_opt_mnist = inertia_mnist.index(min(inertia_mnist[1:])) + 1 if len(inertia_mnist) > 1 else 10
kmeans_mnist = KMeans(n_clusters=k_opt_mnist, random_state=42, n_init=10)
y_pred_mnist = kmeans_mnist.fit_predict(X_mnist_2d)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_mnist_2d[:, 0], X_mnist_2d[:, 1], 
                      c=y_pred_mnist, cmap='tab10', s=10, alpha=0.6)
plt.scatter(kmeans_mnist.cluster_centers_[:, 0], kmeans_mnist.cluster_centers_[:, 1],
            c='black', marker='x', s=200, linewidths=3, label='Центроїди')
plt.title(f'K-means кластеризація MNIST (k={k_opt_mnist}, PCA 2D)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.colorbar(scatter, label='Кластер')
plt.show()

## Висновки

- Для 2D датасету оптимальна кількість кластерів визначається ліктем.
- Для MNIST застосовано PCA для зменшення розмірності до 2D перед кластеризацією та візуалізацією.